# Oppitunti 13 - Agentin muisti Cognee-tietämyspuilla


## Asennus

Tämä muistio näyttää, kuinka rakentaa älykäs **koodausavustaja**, jolla on pysyvä muisti, käyttäen [**Cognee**](https://www.cognee.ai/) tietämyskaavioita ja **Microsoft Agent Frameworkia** (MAF).

Cognee muuntaa jäsentämättömän tekstin rakenteelliseksi, kyseltäväksi tietämyskaavioksi, jota tukevat vektoriupotukset — tarjoten agentillesi monipuolisen, suhteita ymmärtävän pitkäaikaisen muistin.

### Mitä opit
1. **Rakentamaan tietämyskaavioita**: Muunna kehittäjäprofiilit ja parhaat käytännöt rakenteelliseksi, kyseltäväksi tiedoksi.
2. **Integroimaan Cogneen MAF:iin**: Käytä `@tool`-funktioita, jotta MAF-agentti voi kysellä Cogneen tietämyskaaviosta.
3. **Istuntotietoinen keskustelu**: Säilytä konteksti useiden kysymysten yli samassa istunnossa.
4. **Pitkäaikainen muisti**: Säilytä tärkeä tieto istuntojen välillä ja hae se uudessa keskustelussa.

### Edellytykset
- Python 3.9+
- Redis paikallisesti käynnissä (`docker run -d -p 6379:6379 redis`) istunnoin hallintaan
- LLM API-avain (esim. OpenAI) — aseta `LLM_API_KEY` tiedostoon `.env`
- `CACHING=true` tiedostoon `.env` (vaaditaan Cognee-istuntoihin)
- Azure AI Foundry -projekti, jossa on käyttöönotettu chat-malli
- `AZURE_AI_PROJECT_ENDPOINT` ja `AZURE_AI_MODEL_DEPLOYMENT_NAME` tiedostoon `.env`
- Azure CLI kirjautuneena (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## Agentin muistityypit

Tämä muistikirja tutkii samoja kolmea muistityyppiä kuin pää Lesson 13 -muistikirjassa, mutta käyttää Cognee:ta pitkäaikaisena muistiratkaisuna:

| Muistityyppi | Mekanismi | Kesto |
|---|---|---|
| **Työmuisti** | `agent.create_session()` (MAF) | Yksi keskustelu |
| **Lyhytkestoinen** | Cognee istuntovälimuisti (Redis) | Yksi istunto |
| **Pitkäkestoinen** | Cognee tietämyskaavio + vektorit | Pysyvä |

### Cogneen muistirakenne
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## Valmistele Cognee-varasto


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## Osa 1 — Tietopohjan rakentaminen

Keräämme kolmen tyyppistä dataa luodaksemme kattavan tietopohjan koodausavustajallemme:

1. **Kehittäjäprofiili** — henkilökohtainen osaaminen ja tekninen tausta
2. **Pythonin parhaat käytännöt** — Pythonin Zen käytännön ohjeineen
3. **Historialliset keskustelut** — aiemmat kysymys-vastaus -istunnot kehittäjien ja tekoälyavustajien välillä


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## Visualisoi tietämysverkosto

Cognee voi näyttää interaktiivisen HTML-visualisoinnin tunnistamistaan entiteeteistä ja niiden välisistä suhteista.


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## Rikastuta muistia Memifyn avulla

`memify()` analysoi tietämyskarttaa ja luo älykkäitä sääntöjä — tunnistaen malleja, parhaita käytäntöjä ja käsitteiden välisiä suhteita.


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## Osa 2 — MAF-agentti Cognee-työkaluilla

Nyt luomme MAF-agentin, joka voi kysellä Cogneen tietämyskarttaa `@tool`-funktioiden avulla. Tämä antaa agentille mahdollisuuden hyödyntää koko graafitietoista semanttisen haun tehoa säilyttäen samalla keskustelukontekstin sessioiden kautta.


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = await provider.create_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")

## Työmuisti istuntojen kanssa

`AgentSession` (luodaan `agent.create_session()`-kutsulla) tarjoaa työmuistin istunnon aikana. Agentti voi viitata aiempiin viesteihin samalla kun se kysyy Cogneen pitkäaikaista tietoverkkoa.


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## Uusi istunto — Pitkäaikainen muisti säilyy

Uuden istunnon aloittaminen tyhjentää työmuistin, mutta Cogneen tietämysverkosto on edelleen käytettävissä. Agentti voi noutaa saman pitkäaikaisen tiedon täysin uudessa keskustelussa.


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## Yhteenveto

Tässä muistikirjassa rakensit koodausavustajan, joka yhdistää **MAF:n työmuistin** (`agent.create_session()`) ja **Cogneen pitkäaikaisen tietämysgraafin**.

### Mitä opit
1. **Tietämysgraafin rakentaminen**: Cognee käsittelee jäsentämätöntä tekstiä ja rakentaa graafin + vektorimuistin.
2. **Graafin rikastaminen memifyllä**: Johdettujen faktojen ja rikkaampien suhteiden lisääminen olemassa olevaan graafiin.
3. **MAF + Cognee -integraatio**: `@tool`-funktiot sallivat MAF-agentin kysellä Cogneen graafia luonnollisesti.
4. **Työmuisti + pitkäaikainen muisti**: `AgentSession` (kautta `agent.create_session()`) tarjoaa istuntokontekstin, kun taas Cognee tarjoaa pysyvän tiedon.
5. **Suodatettu haku NodeSeteillä**: Kohdenna tiettyihin tietämysgraafin osajoukkoihin (esim. pelkästään periaatteet).

### Keskeiset opit
- **Cognee** muuntaa raakatekstin rakenteelliseksi, suhteita ymmärtäväksi muistiksi — tehokkaampi kuin litteä vektorivarasto.
- **`@tool`-funktiot** yhdistävät MAF-agentit ja ulkoiset tietojärjestelmät selkeästi.
- **`AgentSession`** (kautta `agent.create_session()`) pitää jokaisen keskustelun kontekstin erillään pitkäikäisestä tiedosta.
- Sama tietämysgraafi palvelee useita istuntoja ja agenteja.

### Käytännön sovellukset
- **Kehittäjien kopilotit**: Koodin tarkastus, häiriöanalyysi, arkkitehtiapu
- **Asiakaspalvelukopilotit**: Tukihenkilöt tuotetietojen, UKK:en ja CRM-muistiinpanojen pohjalta
- **Sisäiset asiantuntijakopilotit**: Politiikka-, laki- tai turvallisuusassistentit ohjeistuksia tulkiten
- **Yhtenäiset tietokerrokset**: Yhdistä rakenteellinen ja jäsentämätön data yhdeksi kysyttäväksi graafiksi

### Seuraavat askeleet
- Kokeile ajan tajua Cogneessa
- Määritä OWL-ontologia toimialakohtaisen graafin laadulle
- Lisää käyttäjäpalautekierroksia haun parantamiseksi ajan myötä
- Skaalaa monen agentin järjestelmiksi, jotka jakavat saman Cognee-muistikerroksen


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vastuuvapauslauseke**:
Tämä asiakirja on käännetty käyttämällä tekoälypohjaista käännöspalvelua [Co-op Translator](https://github.com/Azure/co-op-translator). Vaikka pyrimme tarkkuuteen, huomioithan, että automaattikäännöksissä saattaa esiintyä virheitä tai epätarkkuuksia. Alkuperäistä asiakirjaa sen alkuperäiskielellä tulee pitää virallisena lähteenä. Tärkeiden tietojen osalta suositellaan ammattimaista ihmiskäännöstä. Emme ole vastuussa tämän käännöksen käytöstä aiheutuvista väärinymmärryksistä tai tulkinnoista.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
